In [1]:
import numpy as np
import tensorflow as tf
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax.training import train_state

I0000 00:00:1775613909.852657   28573 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775613913.325889   28573 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775613922.892656   28573 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
with tf.io.gfile.GFile("gs://fi2010-lob-data/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt", 'r') as f:
    raw_data = np.loadtxt(f)

In [20]:
start_time = time.time()
X_train = raw_data[:40,:].T
y_train = raw_data[145,:].T - 1
y_train = jnp.array(y_train, dtype=jnp.int32)
print(f'Checking shape: {X_train.shape, y_train.shape}')
print(time.time() - start_time)

Checking shape: ((39512, 40), (39512,))
0.014806032180786133


In [4]:
TIME_STEPS = 127
BATCH_SIZE = 32

dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_train,
    targets=y_train[TIME_STEPS - 1:], 
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    batch_size=BATCH_SIZE,
)
dataset = dataset.unbatch().batch(BATCH_SIZE, drop_remainder=True)

dataset = dataset.prefetch(tf.data.AUTOTUNE)

for batch_x, batch_y in dataset.take(1):
    print(f"抽取出的 X 张量形状: {batch_x.shape}")
    print(f"抽取出的 Y 张量形状: {batch_y.shape},{type(batch_y)}")

E0000 00:00:1775613929.827757   28573 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


抽取出的 X 张量形状: (32, 127, 40)
抽取出的 Y 张量形状: (32,),<class 'tensorflow.python.framework.ops.EagerTensor'>


In [5]:
class TCNBlock(nn.Module):
    features: int
    dilation: int
    kernel_size: int = 3

    @nn.compact 
    def __call__(self, x):
        y = nn.Conv(
            features=self.features,
            kernel_size=(self.kernel_size,),
            kernel_dilation=(self.dilation,),
            padding='CAUSAL',
            dtype = jnp.bfloat16
        )(x)
        return nn.relu(y)

class TCN(nn.Module):
    features: int = 64

    @nn.compact
    def __call__(self, x):
        d = [1, 2, 4, 8, 16, 32]
        
        for dilation in d:
            x = TCNBlock(
                features=self.features, 
                dilation=dilation, 
                kernel_size=3
            )(x)
            
        Output = x[:, -1, :]
        logits = nn.Dense(features=3, dtype = jnp.bfloat16)(Output)
        return logits

In [6]:
def create_train_state(rng, model, learning_rate = 0.01):
    dummy_x = jnp.ones((1, 100, 40)) 
    variables = model.init(rng, dummy_x)
    params = variables['params'] 

    tx = optax.adam(learning_rate)

    return train_state.TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
    )

In [7]:
@jax.jit
def train_step(state, batch_x, batch_y):
    # loss function
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x)
        
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits=logits, labels=batch_y
        ).mean()
        
        preds = jnp.argmax(logits, axis=-1)
        accuracy = jnp.mean(preds == batch_y)
        
        return loss, accuracy 
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    
    (loss, accuracy), grads = grad_fn(state.params)

    state = state.apply_gradients(grads=grads)
    
    return state, loss, accuracy

print("train_step 静态函数编译指令已就绪！")

train_step 静态函数编译指令已就绪！


In [8]:
rng = jax.random.PRNGKey(42) 
tcn_model = TCN()
state = create_train_state(rng, tcn_model, learning_rate=1e-3)

In [9]:
import time

In [ ]:
EPOCHS = 3

print(f"--- 🚀 引擎点火：开始 {EPOCHS} 个 Epoch 的高频 TCN 训练 ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 用于记录当前 Epoch 的累计状态
    epoch_loss = 0.0
    epoch_accuracy = 0.0
    batch_count = 0
    
    # 遍历我们之前做好的 1232 个 Batch 的 tf.data 管道
    for batch_x, batch_y in dataset:
        
        x_jax = jnp.array(batch_x.numpy(), dtype = jnp.bfloat16)
        y_jax = jnp.array(batch_y.numpy(), dtype = jnp.int32)
        
        # 核心爆炸输出：执行单步训练！
        # state 被送进去，伴随着 XLA 机器码的疯狂运算，吐出一个更新了权重的全新 state
        state, loss, accuracy = train_step(state, x_jax, y_jax)
        
        # 累加指标
        epoch_loss += loss
        epoch_accuracy += accuracy
        batch_count += 1
        
        # 打印一下第一个 Batch 的进度，证明计算图已经打通并开始求导
        if batch_count == 1 and epoch == 0:
            print(f"✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: {loss}, Accuracy: {accuracy}")

    # 计算整个 Epoch 的平均指标
    avg_loss = epoch_loss / batch_count
    avg_acc = epoch_accuracy / batch_count
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | 耗时: {epoch_time}s | 平均 Loss: {avg_loss} | 平均 Accuracy: {avg_acc}")

print("--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---")

# Test TCN_model.py

In [11]:
%run TCN_model.py

Loss: 18.088733673095703, Accuracy: 0.0


In [13]:
from TCN_model import TCN, init_train_state, train_step

In [17]:
rng = jax.random.PRNGKey(3721)
rng, dropout_rng = jax.random.split(rng)
model = TCN()
state = init_train_state(rng, model)

In [19]:
for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 用于记录当前 Epoch 的累计状态
    epoch_loss = 0.0
    epoch_accuracy = 0.0
    batch_count = 0
    
    # 遍历我们之前做好的 1232 个 Batch 的 tf.data 管道
    for batch_x, batch_y in dataset:
        
        x_jax = jnp.array(batch_x.numpy(), dtype = jnp.bfloat16)
        y_jax = jnp.array(batch_y.numpy(), dtype = jnp.int32)
        
        # 核心爆炸输出：执行单步训练！
        # state 被送进去，伴随着 XLA 机器码的疯狂运算，吐出一个更新了权重的全新 state
        state, loss, accuracy = train_step(state, x_jax, y_jax, dropout_rng)
        
        # 累加指标
        epoch_loss += loss
        epoch_accuracy += accuracy
        batch_count += 1
        
        # 打印一下第一个 Batch 的进度，证明计算图已经打通并开始求导
        if batch_count == 1 and epoch == 0:
            print(f"✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: {loss}, Accuracy: {accuracy}")

    # 计算整个 Epoch 的平均指标
    avg_loss = epoch_loss / batch_count
    avg_acc = epoch_accuracy / batch_count
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | 耗时: {epoch_time}s | 平均 Loss: {avg_loss} | 平均 Accuracy: {avg_acc}")

print("--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---")

✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: 1.053144931793213, Accuracy: 0.5
Epoch 1/3 | 耗时: 101.54488730430603s | 平均 Loss: 0.94125896692276 | 平均 Accuracy: 0.5556402206420898
Epoch 2/3 | 耗时: 97.05170845985413s | 平均 Loss: 0.9179382920265198 | 平均 Accuracy: 0.5673018097877502
Epoch 3/3 | 耗时: 101.4286859035492s | 平均 Loss: 0.8968228697776794 | 平均 Accuracy: 0.5803861618041992
--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---
